# 71 — Environmental + Documentary Fusion

Combines documentary screening, current target/control outputs, weather readiness, and contradiction state.

In [ ]:
import os, io, re, json, hashlib, platform, sys, zipfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def safe_read_parquet(path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        import pyarrow.parquet as pq
        df = pq.read_table(p).to_pandas()
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "status": "no_rows", "sha256": sha256_file(p)}
        return df, {"path": str(p), "status": "ok", "sha256": sha256_file(p), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def clean_facility(fn: str) -> str:
    s = re.sub(r"\.xlsm$|\.xlsx$|\.pdf$", "", str(fn), flags=re.I)
    s = re.sub(r"^[A-Z0-9]+\s+", "", s)
    s = re.sub(r"\s+Annual Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Annual Performance Report.*$", "", s, flags=re.I)
    s = re.sub(r"\s+Report 2024$", "", s, flags=re.I)
    s = re.sub(r"\s+2024$", "", s)
    return s.strip()

run_cfg = load_yaml(CONFIGS / "run.yml")
phase70_cfg = load_yaml(CONFIGS / "phase70.yml")
doc_cfg = load_yaml(CONFIGS / "documentary_sources.yml")

In [ ]:
step = "71_environmental_documentary_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

catalog, _ = safe_read_csv(DATA / "incinerator_report_catalog.csv")
cems_vocab, _ = safe_read_csv(DATA / "cems_vocab_hits.csv")
cems_blocks, _ = safe_read_csv(DATA / "cems_table_blocks_hits.csv")
emissions, _ = safe_read_parquet(DATA / "emissions_long.parquet")
site_summary, _ = safe_read_csv(OUTPUTS / "42_multi_site_comparison" / "site_summary.csv")
contradictions, _ = safe_read_csv(OUTPUTS / "57_contradictions_red_team_audit" / "contradiction_register.csv")
weather_summary = load_json(OUTPUTS / "52_weather_validation_and_fallback_audit" / "weather_validation_summary.json") if (OUTPUTS / "52_weather_validation_and_fallback_audit" / "weather_validation_summary.json").exists() else {}
news_summary = load_json(OUTPUTS / "53_news_relevance_filtering" / "news_relevance_summary.json") if (OUTPUTS / "53_news_relevance_filtering" / "news_relevance_summary.json").exists() else {}

catalog["permit_id"] = catalog["permit_id"].astype(str)
catalog["facility"] = catalog["filename"].map(clean_facility)
emissions["permit_id"] = emissions["permit_id"].astype(str)
emissions["value_num"] = pd.to_numeric(emissions["value"], errors="coerce")

# documentary score
sum_ex = (
    emissions[emissions["metric"].astype(str).str.contains("exceedance_count", case=False, na=False)]
    .groupby(["permit_id", "pollutant"], as_index=False)["value_num"].max()
)
expermit = sum_ex.groupby("permit_id", as_index=False)["value_num"].sum().rename(columns={"value_num": "exceedance_total"})

cems_vocab["permit_id"] = cems_vocab["permit_id"].astype(str)
cems_vocab["txt"] = (cems_vocab.get("key","").astype(str) + " " + cems_vocab.get("value_text","").astype(str)).str.lower()
cems_vocab["noncomp"] = cems_vocab["txt"].str.contains("non-compliance|non compliance|exceed|exceedance|breach|limit", regex=True)
cv_sum = cems_vocab.groupby("permit_id").agg(
    cems_vocab_hits=("permit_id", "size"),
    noncomp_hits=("noncomp", "sum"),
).reset_index()

cems_blocks["permit_id"] = cems_blocks["permit_id"].astype(str)
cb_sum = cems_blocks.groupby("permit_id").size().reset_index(name="cems_block_hits")

base_em = emissions[["permit_id", "pollutant", "metric", "value_num"]].copy()
def pickmax(df, metric):
    return (
        df[df["metric"].astype(str) == metric]
        .groupby(["permit_id", "pollutant"])["value_num"]
        .max()
        .rename(metric)
    )

rat = pd.concat([pickmax(base_em, "ELV"), pickmax(base_em, "daily_mean"), pickmax(base_em, "half_hourly_mean"), pickmax(base_em, "p95"), pickmax(base_em, "max")], axis=1).reset_index()
for metric in ["daily_mean", "half_hourly_mean", "p95", "max"]:
    rat[f"{metric}_ratio"] = rat[metric] / rat["ELV"]
rat["worst_ratio"] = rat[[c for c in rat.columns if c.endswith("_ratio")]].max(axis=1, skipna=True)
ratp = rat.groupby("permit_id", as_index=False)["worst_ratio"].max()

rank = (
    catalog[["permit_id", "facility"]]
    .drop_duplicates()
    .merge(cv_sum, on="permit_id", how="left")
    .merge(cb_sum, on="permit_id", how="left")
    .merge(expermit, on="permit_id", how="left")
    .merge(ratp, on="permit_id", how="left")
)

for col in ["cems_vocab_hits","noncomp_hits","cems_block_hits","exceedance_total","worst_ratio"]:
    rank[col] = pd.to_numeric(rank[col], errors="coerce").fillna(0)

rank["documentary_risk_score"] = rank["exceedance_total"] * 3 + rank["noncomp_hits"] * 0.2 + rank["worst_ratio"] * 10 + rank["cems_block_hits"] * 0.01
rank["weather_ready"] = bool(weather_summary.get("weather_ready", False))
rank["news_kept_count"] = int(news_summary.get("kept_article_count", 0))
rank["contradiction_count"] = int(len(contradictions)) if not contradictions.empty else 0
rank["environmental_signal_score"] = 0.0
if not site_summary.empty:
    rank["environmental_signal_score"] = float(pd.to_numeric(site_summary["mean_site_score"], errors="coerce").fillna(0).max())
rank["integrated_screening_score"] = rank["documentary_risk_score"] + rank["environmental_signal_score"] - rank["contradiction_count"] * 0.05
rank = rank.sort_values(["integrated_screening_score","documentary_risk_score"], ascending=False)
rank.to_csv(out / "integrated_screening_table.csv", index=False)

summary = {
    "rows": int(len(rank)),
    "weather_ready": bool(weather_summary.get("weather_ready", False)),
    "news_kept_count": int(news_summary.get("kept_article_count", 0)),
    "contradiction_count": int(len(contradictions)) if not contradictions.empty else 0,
}
write_json(out / "integrated_screening_summary.json", summary)
manifest = manifest_base(step, [CONFIGS / "phase70.yml", CONFIGS / "documentary_sources.yml"])
add_artifact(manifest, out / "integrated_screening_table.csv")
add_artifact(manifest, out / "integrated_screening_summary.json")
write_json(out / "manifest.json", manifest)
display(rank.head(20))